# 04.03 — Case Study: Mini RAG — QA dengan Konteks

**Tujuan**: bangun **mini RAG** (Retrieval Augmented Generation): retrieve chunk relevan dari knowledge base, lalu generate jawaban. Bandingkan Groq vs TinyLlama Q4 ketika dua-duanya dapat konteks yang sama.

**Hipotesis**: dengan konteks yang relevan, **SLM jadi viable** untuk RAG — karena model tidak perlu "tahu segala", cuma perlu menyusun jawaban dari konteks.

**Prasyarat**: modul 02 lulus (TinyLlama GGUF sudah di-download).

**Komponen mini-RAG**:
- Knowledge base: 5 paragraf tentang topik Indonesia (sengaja dibuat sederhana)
- Retrieval: TF-IDF + cosine similarity (sklearn)
- Generation: Groq + TinyLlama

## 0. Bootstrap (jalankan pertama)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_NAME = "llm-vs-slm-lab"
    REPO_URL = "https://github.com/rizkyhaksono/llm-vs-slm-lab.git"
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -r requirements.txt

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "requirements.txt").exists():
        repo_root = candidate
        break
assert repo_root is not None
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"IN_COLAB={IN_COLAB}, repo_root={repo_root}")

## 1. Knowledge base

5 paragraf — sengaja **mengandung fakta spesifik** yang tidak ada di training data umum LLM (atau detail-detail yang bisa di-validate).

In [ ]:
KNOWLEDGE_BASE = [
    "Borobudur adalah candi Buddha terbesar di dunia yang terletak di Magelang, Jawa Tengah. "
    "Candi ini dibangun pada abad ke-8 hingga ke-9 Masehi pada masa Dinasti Syailendra. "
    "Struktur Borobudur terdiri dari 9 platform bertingkat (6 persegi dan 3 lingkaran) dan dihiasi 2.672 panel relief.",

    "Rendang adalah masakan tradisional Minangkabau dari Sumatera Barat yang dibuat dengan cara memasak daging sapi "
    "dalam santan kelapa dan campuran rempah selama 4 sampai 8 jam hingga kering. Pada tahun 2011, CNN International "
    "menobatkan rendang sebagai makanan terenak nomor 1 di dunia dalam survei pembacanya.",

    "Pulau Komodo terletak di Nusa Tenggara Timur dan menjadi habitat alami dari kadal raksasa Komodo (Varanus komodoensis), "
    "reptil terbesar di dunia yang panjangnya mencapai 3 meter dan beratnya hingga 70 kilogram. Taman Nasional Komodo "
    "didirikan tahun 1980 dan masuk dalam daftar UNESCO World Heritage sejak 1991.",

    "Wayang kulit adalah seni pertunjukan tradisional Jawa yang menggunakan boneka pipih terbuat dari kulit kerbau "
    "yang diproyeksikan pada layar. Dalang adalah pengendali wayang yang juga bertugas menyanyi, bercerita, dan mengiringi "
    "orkestra gamelan. UNESCO mengakui wayang sebagai Masterpiece of the Oral and Intangible Heritage of Humanity tahun 2003.",

    "Gunung Bromo adalah gunung berapi aktif setinggi 2.329 meter di atas permukaan laut yang terletak di Jawa Timur. "
    "Gunung ini dikelilingi oleh lautan pasir bernama Tengger seluas sekitar 10 kilometer persegi. Suku Tengger yang tinggal di sekitar "
    "Bromo merayakan upacara Yadnya Kasada setiap tahun dengan melempar sesaji ke kawah gunung.",
]
print(f"Knowledge base: {len(KNOWLEDGE_BASE)} chunks")

## 2. Retrieval: TF-IDF + cosine similarity

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=1)
kb_vectors = vectorizer.fit_transform(KNOWLEDGE_BASE)

def retrieve(question: str, top_k: int = 1) -> list[tuple[int, float, str]]:
    q_vec = vectorizer.transform([question])
    sims = cosine_similarity(q_vec, kb_vectors)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(int(i), float(sims[i]), KNOWLEDGE_BASE[i]) for i in top_idx]

# Test retrieval
for q in ["Berapa platform Borobudur?", "Apa makanan Minang terkenal?", "Tinggi Gunung Bromo?"]:
    hits = retrieve(q, top_k=1)
    idx, sim, chunk = hits[0]
    print(f"Q: {q}")
    print(f"  → chunk {idx} (sim={sim:.3f}): {chunk[:80]}...\n")

## 3. Generation dengan 2 backend

Prompt: "Berdasarkan konteks berikut, jawab pertanyaan. Kalau jawaban tidak ada di konteks, bilang tidak tahu."

In [ ]:
import time
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

from utils.llm_clients import GROQ_DEFAULT_MODEL, groq_client

groq = groq_client()

gguf_path = hf_hub_download(
    repo_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
    filename="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
    local_dir=str(repo_root / "models"),
)
tiny = Llama(model_path=gguf_path, n_ctx=2048, n_threads=4, verbose=False)

RAG_PROMPT = (
    "Jawab pertanyaan berdasarkan KONTEKS di bawah. "
    "Kalau jawaban tidak ada di konteks, jawab 'Tidak ada di konteks'. "
    "Jawab singkat dalam Bahasa Indonesia.\n\n"
    "KONTEKS:\n{context}\n\n"
    "PERTANYAAN: {question}\n"
    "JAWABAN:"
)

def answer_groq(question: str, context: str) -> tuple[str, float]:
    t0 = time.perf_counter()
    r = groq.chat.completions.create(
        model=GROQ_DEFAULT_MODEL,
        messages=[{"role": "user", "content": RAG_PROMPT.format(context=context, question=question)}],
        max_tokens=80, temperature=0,
    )
    ms = (time.perf_counter() - t0) * 1000
    return r.choices[0].message.content.strip(), ms

def answer_tiny(question: str, context: str) -> tuple[str, float]:
    t0 = time.perf_counter()
    r = tiny.create_chat_completion(
        messages=[{"role": "user", "content": RAG_PROMPT.format(context=context, question=question)}],
        max_tokens=80, temperature=0,
    )
    ms = (time.perf_counter() - t0) * 1000
    return r["choices"][0]["message"]["content"].strip(), ms

print("Backends ready.")

## 4. Jalankan 6 pertanyaan

Mix: 5 pertanyaan yang ada di knowledge base + 1 pertanyaan trick (di luar KB) untuk test apakah model mau bilang "tidak tahu".

In [ ]:
QUESTIONS = [
    "Borobudur memiliki berapa panel relief?",
    "Apa peringkat rendang dalam survei CNN tahun 2011?",
    "Berapa panjang maksimal kadal Komodo?",
    "Tahun berapa wayang diakui UNESCO?",
    "Berapa tinggi Gunung Bromo?",
    "Siapa presiden Indonesia tahun 2026?",  # trick — tidak ada di KB
]

results = []
for q in QUESTIONS:
    hits = retrieve(q, top_k=1)
    _, sim, context = hits[0]
    groq_ans, groq_ms = answer_groq(q, context)
    tiny_ans, tiny_ms = answer_tiny(q, context)
    results.append({
        "question": q,
        "retrieval_sim": round(sim, 3),
        "context_preview": context[:80] + "...",
        "groq_ans": groq_ans, "groq_ms": round(groq_ms),
        "tiny_ans": tiny_ans, "tiny_ms": round(tiny_ms),
    })
print(f"Done — {len(results)} questions answered.")

## 5. Display side-by-side

In [ ]:
for r in results:
    print("=" * 80)
    print(f"Q: {r['question']}")
    print(f"  Retrieved (sim={r['retrieval_sim']}): {r['context_preview']}")
    print(f"\n  [GROQ — {r['groq_ms']} ms]")
    print(f"  {r['groq_ans']}")
    print(f"\n  [TINYLLAMA Q4 — {r['tiny_ms']} ms]")
    print(f"  {r['tiny_ans']}")
print("=" * 80)

## 6. Tabel ringkas

In [ ]:
import pandas as pd
df = pd.DataFrame([
    {
        "q": r["question"][:40],
        "groq": r["groq_ans"][:80],
        "tiny": r["tiny_ans"][:80],
        "groq_ms": r["groq_ms"], "tiny_ms": r["tiny_ms"],
    }
    for r in results
])
df

## Refleksi & insight

1. **Dengan konteks relevan, TinyLlama Q4 sering cukup baik** untuk extractive question (fakta literal dari KB). Itu **insight kunci**: kualitas SLM melonjak signifikan ketika dia tidak perlu "mengingat" — cuma menyusun dari konteks.
2. **Trick question ("presiden 2026")**: Groq biasanya bilang "tidak ada di konteks" (mengikuti instruksi). TinyLlama sering hallucinate jawaban — itu kelemahan model kecil.
3. **Retrieval quality matters**: TF-IDF di sini sangat sederhana. Untuk production, pakai embedding (sentence-transformers, OpenAI ada-002, dll) untuk semantic search.
4. **RAG kombinasikan strength dua-duanya**: knowledge fresh & verifiable dari KB, fluency natural dari LLM. Standar arsitektur untuk Q&A internal company.
5. **Implikasi practical**: untuk RAG over private docs (kontrak, dokumentasi internal), **SLM lokal jadi opsi nyata** — privacy aman, biaya nol, kualitas cukup.

## Latihan mandiri

1. Ganti retrieval ke `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (semantic). Apakah quality retrieval naik untuk pertanyaan yang paraphrase?
2. Tambah `top_k=3` di retrieval — gabungkan 3 chunk jadi konteks. Apakah jawaban lebih akurat atau model malah bingung?
3. Coba pertanyaan yang **butuh reasoning multi-step** (gabungan 2 chunk), mis. "Mana yang lebih tua: Borobudur atau Komodo National Park?". Apakah TinyLlama bisa atau cuma Groq?

## Lanjut

Selesai modul case study. Sekarang formalisasi: kapan pakai apa di proyek nyata: [../05-kapan-pakai-apa/01_decision_framework.ipynb](../05-kapan-pakai-apa/01_decision_framework.ipynb)